### Code libraries to review before operating (in the following order)<br>
1. intro_to_agents.agents.llms
2. intro_to_agents.text_extractors
3. intro_to_agents.chunkers
4. intro_to_agents.embedders
5. intro_to_agents.vector_databases

### Imports

In [ ]:
# Imports
import os

from dotenv import load_dotenv

from intro_to_agents.rag.text_extractors import MarkitdownExtractor, TextExtractor, load_file_paths_from_folder
from intro_to_agents.rag.chunkers import SplitCharChunker, CharLenChunker, SentenceChunker, SemanticChunker
from intro_to_agents.rag.embedders import SentenceTransformerEmbedder
from intro_to_agents.agents.llms import OpenAILLM
from intro_to_agents.rag.vector_databases import ChromaDBVectorDB

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Make sure you have a .env file with your OpenAI API key
    # ex: OPENAI_API_KEY="abcdefg12345"
    # a .env file is a text file that contains environment variables
    # it is used to store sensitive information (like API keys) in a secure way
    # it is NOT included in the repository (you don't want to share your API key with others), so you will need to create it yourself

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-5-mini"
llm = OpenAILLM(api_key=api_key, model=model)

## Example of Process

In [ ]:
# Identify file or folder full of documents to chunk
# * This folder can contain any number of documents, and they can be of any type (e.g. .txt, .pdf, .docx, .pptx, etc.)

# @ You: Identify the folder containing your documents
folder = "../data/text_corpus"

# Load all file paths in a folder
filepaths = load_file_paths_from_folder(folder)

# Extract text 
# text_extractor = TextExtractor()
text_extractor = MarkitdownExtractor()
text = text_extractor.extract(filepaths)

In [ ]:
# Test text extraction
print(text[0])

In [ ]:
# Chunk documents
    # @ You: Choose your chunker: SplitCharChunker, CharLenChunker, SentenceChunker, SemanticChunker
    # Note: SentenceChunker takes window_size (default 5), not llm. Use SemanticChunker(llm) for semantic chunking.

chunker = SemanticChunker(llm)
# chunker = SentenceChunker(window_size=5)

# Chunk the first document in our corpus (text[0])
docs = chunker.chunk(text[0])

# * Visualize resultant chunks (unnecessary; just here for you to see what it's doing)
n = len(docs[0])
k = 10
if n < k:
    k = n
for i in range(k):
    print(docs[0][i].rstrip(), "\n", "-"*100)

In [ ]:
chunker = SentenceChunker(window_size=2)

# Chunk the first document in our corpus (text[0])
docs = chunker.chunk(text[0])

# * Visualize resultant chunks (unnecessary; just here for you to see what it's doing)
n = len(docs[0])
k = 10
if n < k:
    k = n
for i in range(k):
    print(docs[0][i].rstrip(), "\n", "-"*100)

In [ ]:
# Embed chunks
embedder = SentenceTransformerEmbedder()
embeddings = embedder.embed(docs)

# * Visualize embedding of first chunk of first document (unnecessary; just here for you to see what it's doing)
print(embeddings[0][0])

## Actual Code to Execute

In [ ]:
# Load to vector database
# @ You: Identify the path where you want to save your vector database
dbpath = r"C:\Users\sandidgeh\Downloads"

# @ You: Identify the folder containing your documents
folder = "../data/text_corpus"

# Load all file paths in a folder
filepaths = load_file_paths_from_folder(folder)

# Extract, chunk, and create embeddings
text_extractor = MarkitdownExtractor()
chunker = SentenceChunker(window_size=2) # @You: put whatever chunker you want here
embedder = SentenceTransformerEmbedder()

# Create a vector database
vdb = ChromaDBVectorDB(dbpath = dbpath, 
                       text_extractor = text_extractor,
                       chunker = chunker, 
                       embedder = embedder, 
                       distance_measure = "cosine")

# Connect to vector database
vdb.initialize_db()

# Create a collection (think of this like a table in a SQL database)
# ! DO NOT LOSE THE NAME OF THIS COLLECTION!!! You will need it to be able to retrieve documents from the vector database
collection_name = "TGNSI_Commerce_Letters"
vdb.initialize_collection(collection_name=collection_name)

# Add documents to vector database
vdb.add_to_collection(filepaths)

In [ ]:
# Test: Retrieve documents from vector database
docs, distances = vdb.retrieve("What agricultural products does TGNSI get from Sweden?", k = 3)

# These results are what we will use to augument the original query (injecting private corpus into the LLM that it has absolutely no knowledge of in its weights (think LLM brain))
for i, (doc, distance) in enumerate(zip(docs, distances)):
    print(f"Document {i+1}:")
    print(f"Distance: {distance:.4f}")
    print(f"Content: {doc}\n")